 # Smart Factory Sensor Data Analysis


 # Claudio Bayro

In [1]:
!pip install pyspark

  Using cached pyspark-4.1.1.tar.gz (455.4 MB)
  Preparing metadata (setup.py) ... done
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008663 sha256=6b523e7b139b21f42df15bf1ba043041ac14cf51a9dd401d6a7cd5209daa4821
  Stored in directory: /root/.cache/pip/wheels/16/77/d3/d15aaaab1df8384ad9bd94caba26a1a5aa439d8afd187a5ab9
Successfully built pyspark


In [2]:
from pyspark.sql import SparkSession
from datetime import datetime

In [4]:
spark = SparkSession.builder \
    .appName("Smart Factory Sensor Data Analysis") \
    .getOrCreate()

In [5]:
factory_data = [
    ("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
    ("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
    ("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
    ("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
    ("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
    ("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
    ("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
    ("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
    ("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
    ("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

In [6]:
df = spark.createDataFrame(factory_data, ["machine_id", "timestamp", "temperature"])

In [7]:
df.createOrReplaceTempView("factory_sensors")

# 1. Explore the schema of the DataFrame

In [8]:
df.printSchema()

root
 |-- machine_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature: double (nullable = true)



# 2. Get the average temperature per machine



In [10]:
spark.sql("""
    SELECT 
        machine_id,
        AVG(temperature) AS avg_temperature
    FROM factory_sensors
    GROUP BY machine_id
    ORDER BY machine_id
""").show()

+----------+-----------------+
|machine_id|  avg_temperature|
+----------+-----------------+
|      M001|           76.725|
|      M002|69.53333333333333|
|      M003|             73.4|
+----------+-----------------+



In [11]:
df.createOrReplaceTempView("factory_sensors")

# 3. Find the maximum and minimum temperature per machine

In [12]:
spark.sql("""
    SELECT 
        machine_id,
        MAX(temperature) AS max_temperature,
        MIN(temperature) AS min_temperature
    FROM factory_sensors
    GROUP BY machine_id
    ORDER BY machine_id
""").show()

+----------+---------------+---------------+
|machine_id|max_temperature|min_temperature|
+----------+---------------+---------------+
|      M001|           78.0|           75.3|
|      M002|           70.1|           68.7|
|      M003|           74.6|           72.4|
+----------+---------------+---------------+



# 4. Filter records above a temperature threshold (temp > 75)

In [13]:
spark.sql("""
    SELECT *
    FROM factory_sensors
    WHERE temperature > 75
    ORDER BY timestamp
""").show()

+----------+-------------------+-----------+
|machine_id|          timestamp|temperature|
+----------+-------------------+-----------+
|      M001|2026-01-26 08:00:00|       75.3|
|      M001|2026-01-26 08:10:00|       76.1|
|      M001|2026-01-26 08:25:00|       77.5|
|      M001|2026-01-26 08:40:00|       78.0|
+----------+-------------------+-----------+



# 5. Count the number of readings per machine

In [14]:
spark.sql("""
    SELECT 
        machine_id,
        COUNT(*) AS reading_count
    FROM factory_sensors
    GROUP BY machine_id
    ORDER BY machine_id
""").show()

+----------+-------------+
|machine_id|reading_count|
+----------+-------------+
|      M001|            4|
|      M002|            3|
|      M003|            3|
+----------+-------------+



# 6. Find the machine with the highest temperature.

In [15]:
spark.sql("""
    SELECT machine_id, timestamp, temperature
    FROM factory_sensors
    ORDER BY temperature DESC
    LIMIT 1
""").show()


+----------+-------------------+-----------+
|machine_id|          timestamp|temperature|
+----------+-------------------+-----------+
|      M001|2026-01-26 08:40:00|       78.0|
+----------+-------------------+-----------+

